# ViajeExpress payment intelligence

Run all cells from this project directory. Python 3.10+; the notebook requires an existing Jupyter/IPython installation. The standalone dashboard needs no notebook dependencies. All data is synthetic. See README.md for contracts, metrics, and limitations.

In [ ]:
from pathlib import Path
import json, sqlite3
from contextlib import closing
from run import build
ROOT = Path.cwd()
assert (ROOT / "payments").is_dir(), "Open this notebook from the viajeexpress project directory"
bundle = build(ROOT / "data/raw", ROOT / "data/processed")
bundle["quality"]

## Trace messy records into the unified schema
Inspect one original JSON payload, a normalized row and a rejected row. Monetary amounts are integer minor units; timestamps are UTC.

In [ ]:
raw = json.loads((ROOT / "data/raw/psp1.json").read_text(encoding="utf-8"))
print("Source row:", json.dumps(raw[0], indent=2))
print("Unified row:", json.dumps(next(r for r in bundle["transactions"] if r["id"] == "PSP1:txn-00001"), indent=2))
rejected = json.loads((ROOT / "data/processed/quarantine.json").read_text(encoding="utf-8"))
print("Quarantined:", json.dumps(rejected[0], indent=2))

## Interactive dashboard
Trust this notebook to permit JavaScript in the embedded dashboard. It includes all four filters, currency-separated trends, cost comparisons and CSV export. If your notebook frontend blocks iframe scripts, open data/processed/dashboard.html directly or use python run.py --serve.

In [ ]:
from IPython.display import HTML, display
from html import escape
html = (ROOT / "data/processed/dashboard.html").read_text(encoding="utf-8")
display(HTML('<iframe title="ViajeExpress payment analytics" style="width:100%;height:1250px;border:0" srcdoc="' + escape(html, quote=True) + '"></iframe>'))

## Query the ledger directly
The example separates card authorization from asynchronous payment outcomes. Pending payments are excluded; fully refunded payments imply an earlier success under the source contracts.

In [ ]:
query = """SELECT psp, country, COUNT(*) AS final_card_decisions,
 ROUND(100.0 * SUM(status IN ('approved','refunded')) / COUNT(*), 2) AS card_auth_pct
 FROM transactions WHERE method IN ('credit_card','debit_card')
 AND status IN ('approved','declined','refunded')
 GROUP BY psp,country ORDER BY psp,country"""
with closing(sqlite3.connect(ROOT / "data/processed/payments.sqlite")) as db:
    for row in db.execute(query):
        print(row)

## Weekly summary

In [ ]:
from IPython.display import Markdown
display(Markdown((ROOT / "data/processed/weekly-report.md").read_text(encoding="utf-8")))